In [1]:
from transformers import AutoTokenizer,AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [2]:
base_model_name = "mistralai/Mistral-7B-v0.1"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
def tokenize_fxn(examples):
  tokens = tokenizer(examples['text'], padding='max_length', truncation=True,
                     max_length=512)
  tokens['labels'] = tokens['input_ids'].copy()
  return tokens

In [5]:
dataset = load_dataset("json", data_files="/content/ds001.jsonl")

In [37]:
dataset['train'][78]

{'text': '1 data collection in this study, we used finger print blood group dataset publicly available on kaggle'}

In [6]:
tokenized_dataset = dataset.map(tokenize_fxn, batched=True, remove_columns=["text"])

In [7]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 158
    })
})

In [8]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [10]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True)

qmodel = AutoModelForCausalLM.from_pretrained(base_model_name,
                                              quantization_config=bnb_config,
                                              device_map="auto")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [11]:
model = get_peft_model(qmodel, lora_config)

In [12]:
from transformers import TrainingArguments, Trainer

In [14]:
split_dataset = tokenized_dataset['train'].train_test_split(test_size=0.1)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [21]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [22]:
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    do_eval=True,
    eval_steps=50,
    logging_steps=20,
    save_total_limit=1,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

In [23]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
20,0.188748
40,0.161165
60,0.165383
80,0.159895
100,0.139167
120,0.186240
140,0.142603
160,0.138561
180,0.144603
200,0.135189


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


TrainOutput(global_step=710, training_loss=0.12071721427877184, metrics={'train_runtime': 1967.8561, 'train_samples_per_second': 0.361, 'train_steps_per_second': 0.361, 'total_flos': 1.551663592636416e+16, 'train_loss': 0.12071721427877184, 'epoch': 5.0})

In [39]:
prompt = "there are several studies that shows the correlation"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


there are several studies that shows the correlation between blood group and certain fingerprint features, which makes it possible to identify blood groups using fingerprints


In [43]:
# import torch

# torch.cuda.empty_cache()

# model.eval()
# total_loss = 0
# count = 0

# for batch in eval_dataset:
#     inputs = torch.tensor(batch["input_ids"]).unsqueeze(0).to(model.device)
#     mask = torch.tensor(batch["attention_mask"]).unsqueeze(0).to(model.device)
#     labels = inputs.clone()

#     with torch.no_grad():
#         outputs = model(input_ids=inputs, attention_mask=mask, labels=labels)
#         total_loss += outputs.loss.item()
#         count += 1

# print("Loss:", total_loss / count)